In [1]:
import tkinter as tk
from tkinter import messagebox, simpledialog
import json
import os

class Contact:
    def __init__(self, name, phone, email):
        self.name = name
        self.phone = phone
        self.email = email

    def to_dict(self):
        return {'name': self.name, 'phone': self.phone, 'email': self.email}

    @staticmethod
    def from_dict(data):
        return Contact(data['name'], data['phone'], data['email'])

    def __str__(self):
        return f"{self.name} - {self.phone} - {self.email}"

class ContactBookApp:
    def __init__(self, master):
        self.master = master
        self.master.title("Contact Book")
        self.filename = "contacts.json"
        self.contacts = self.load_contacts()

        # GUI Layout
        self.contact_listbox = tk.Listbox(master, width=50)
        self.contact_listbox.pack(pady=10)

        self.add_button = tk.Button(master, text="Add Contact", command=self.add_contact)
        self.add_button.pack()

        self.delete_button = tk.Button(master, text="Delete Selected", command=self.delete_contact)
        self.delete_button.pack()

        self.search_button = tk.Button(master, text="Search Contact", command=self.search_contact)
        self.search_button.pack()

        self.refresh_contacts()

    def save_contacts(self):
        with open(self.filename, "w") as f:
            json.dump([c.to_dict() for c in self.contacts], f)

    def load_contacts(self):
        if os.path.exists(self.filename):
            with open(self.filename, "r") as f:
                try:
                    data = json.load(f)
                    return [Contact.from_dict(item) for item in data]
                except json.JSONDecodeError:
                    return []
        return []

    def refresh_contacts(self):
        self.contact_listbox.delete(0, tk.END)
        for contact in self.contacts:
            self.contact_listbox.insert(tk.END, str(contact))

    def add_contact(self):
        name = simpledialog.askstring("Name", "Enter contact name:")
        if not name: return
        phone = simpledialog.askstring("Phone", "Enter contact phone:")
        email = simpledialog.askstring("Email", "Enter contact email:")

        contact = Contact(name, phone, email)
        self.contacts.append(contact)
        self.save_contacts()
        self.refresh_contacts()

    def delete_contact(self):
        selected = self.contact_listbox.curselection()
        if not selected:
            messagebox.showinfo("Info", "Please select a contact to delete.")
            return
        index = selected[0]
        del self.contacts[index]
        self.save_contacts()
        self.refresh_contacts()

    def search_contact(self):
        name = simpledialog.askstring("Search", "Enter name to search:")
        if not name: return

        for i, contact in enumerate(self.contacts):
            if contact.name.lower() == name.lower():
                self.contact_listbox.selection_clear(0, tk.END)
                self.contact_listbox.selection_set(i)
                self.contact_listbox.see(i)
                return
        messagebox.showinfo("Not Found", "Contact not found.")

# Launch app
if __name__ == "__main__":
    root = tk.Tk()
    app = ContactBookApp(root)
    root.mainloop()
